Connexion à l'API

In [ ]:
import requests
import os
from dotenv import load_dotenv

# 1. Charger le coffre-fort (.env)
load_dotenv()

# 2. Récupérer la clé secrète
API_KEY = os.getenv('OPENWEATHER_API_KEY')

CITY = 'Casablanca'
URL = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"

response = requests.get(URL)

if response.status_code == 200:
    data = response.json()
    print("✅ Données récupérées avec succès")
else:
    print("❌ Erreur lors de la requête API :", response.status_code)

Extraction des données

In [ ]:
weather_data = {
    'Ville': data['name'],
    'Temperature': data['main']['temp'],
    'Temp_min': data['main']['temp_min'],
    'Temp_max': data['main']['temp_max'],
    'Pression': data['main']['pressure'],
    'Humidité': data['main']['humidity'],
    'Description': data['weather'][0]['description'],
    'Vent_Vitesse':data['wind']['speed']
}
print(weather_data)

Sauvegarde dans un fichier CSV

In [ ]:
import pandas as pd

df = pd.DataFrame([weather_data])
df.to_csv(data['name']+'_weather.csv', index=False)
print("fichier CSV cree avec succees")
df

Meteo de plusieurs villes

In [ ]:
cities=['Casablanca','Manchester', 'Berlin', 'Tokyo', 'New York']
all_data=[]
for city in cities:
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"
    rep = requests.get(url)
    if rep.status_code == 200:
        dt = rep.json()
        all_data.append({
            'Ville': dt['name'],
            'Temperature': dt['main']['temp'],
            'Humidité': dt['main']['humidity'],
            'Description': dt['weather'][0]['description'],
        })
datas = pd.DataFrame(all_data)
datas.to_csv('cities_weather.csv', index=False)
print("fichier CSV pour les villes cree avec succees")
datas

In [ ]:
import schedule
import time
from datetime import datetime

def collect_weather() :
    response = requests.get(URL)
    if response.status_code == 200 :
        data = response.json()
        weather_data = {
            'Datetime' : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'Ville' : data['main'],
            'Temperature' : data['main']['temp'],
            'Humidité' : data['main']['humidity'],
            'Description' : data['weather'][0]['description']
        }
        df = pd.DataFrame([weather_data])
        df.to_csv('meteo_automatique.csv', mode='a', header=not pd.io.common.file_exists('meteo_automatique.csv'), index=False)
        print(f"Donnees collectees a {weather_data['Datetime']}")
schedule.every(1).minutes.do(collect_weather)
print("lancement de la collecte horaire ...")
while True:
    schedule.run_pending()
    time.sleep(1)

Partie 2 : Collecte depuis une BD relationnelle

1. Connexion à la base de données 


In [ ]:
from sqlalchemy import create_engine
import pandas as pd

USER = 'postgres'
PASSWORD = 'admin'
HOST = 'localhost'
PORT = '5432'
DB = 'transport_meteo'
engine = create_engine(f'postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB}')

2. Extraction des données jointes 


In [ ]:
query = '''
SELECT tb.date , tb.ligne , tb.arret , tb.nombre_passagers , m.temperature ,m.pluie , m.vent , m.description
FROM trafic_bus tb
JOIN meteo m ON tb.date = m.date
ORDER BY tb.date , tb.ligne ;
'''
df=pd.read_sql(query,engine)
df

3. Export CSV 


In [ ]:
df.to_csv('trafic_meteo.csv',index=False)
print("Export reussi des donnees dans trafic_meteo.csv")

4. Automatisation toutes les heures 


In [ ]:
import schedule
import time
from datetime import datetime 

def extract_data():
  df = pd.read_sql(query, engine)
  filename = f"transport_meteo{datetime.now().strftime('%Y-%m-%d_%H_%M_%S')}.csv"
  df.to_csv(filename, index=False)
  print(f"donnees importes avec succes dans {filename}")

schedule.every().hour.do(extract_data)
print("Extraction planifiée toutes les heures...")
while True:
    schedule.run_pending()
    time.sleep(1)
